# 02 · Preprocesamiento → Feature Store
### Prueba final MLOps · Cesar Romero

**Qué exige el examen en esta etapa**
- Realizar las transformaciones necesarias sobre los datos.
- Registrar las variables generadas usando **Feature Store**.

**Entrada:** `cancer_raw` (notebook 01)
**Salida:** Feature Table `<catalog>.mlops_final.cancer_features`

> **Concepto clave:** la Feature Table guarda la **entidad + sus variables**, nunca la etiqueta.
> El `target` se queda en `cancer_raw`. Esto evita *label leakage* y permite reutilizar
> las mismas features para otros modelos con otras etiquetas.

## 0. Librería de Feature Engineering

El cliente de Feature Store de Unity Catalog vive en `databricks-feature-engineering`
(sucesor de `databricks-feature-store`). `%restart_python` es obligatorio tras el pip install
para que el import quede disponible.

In [ ]:
%pip install databricks-feature-engineering --quiet
%restart_python

## 1. Configuración (mismo contrato que el notebook 01)

In [ ]:
CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
SCHEMA  = "mlops_final"

RAW_TABLE     = f"{CATALOG}.{SCHEMA}.cancer_raw"
FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.cancer_features"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
print("Raw      :", RAW_TABLE)
print("Features :", FEATURE_TABLE)

## 2. Transformaciones

Tres features derivadas, todas construidas con funciones Spark (no pandas) para que
el cálculo sea distribuido y reproducible:

| Feature | Fórmula | Por qué |
|---|---|---|
| `area_radius_ratio` | `area / radius²` | Mide cuánto se desvía el núcleo de un círculo perfecto. Un círculo da ≈ π. |
| `perimeter_radius_ratio` | `perimeter / radius` | Ídem, versión lineal (≈ 2π en un círculo). Irregularidad del contorno. |
| `compactness_x_texture` | `compactness * texture` | Interacción: combina irregularidad con heterogeneidad de grises. |

**Decisión de diseño:** el escalado (`StandardScaler`) **NO** se hace aquí. Va dentro del
`Pipeline` de sklearn en el notebook 03. Si escalara ahora, el endpoint REST recibiría
datos crudos y el modelo predeciría mal — el preprocesamiento tiene que viajar con el modelo.

In [ ]:
from pyspark.sql import functions as F

raw_df = spark.table(RAW_TABLE)

features_df = (
    raw_df
      .withColumn("area_radius_ratio",      F.col("area") / (F.col("radius") ** 2))
      .withColumn("perimeter_radius_ratio", F.col("perimeter") / F.col("radius"))
      .withColumn("compactness_x_texture",  F.col("compactness") * F.col("texture"))
      .drop("target")                      # la etiqueta NO entra al Feature Store
)

FEATURE_COLS = [c for c in features_df.columns if c != "patient_id"]
print("Features publicadas:", FEATURE_COLS)
display(features_df.limit(10))

## 3. Control de calidad previo al registro

Antes de publicar: nulos y duplicados en la clave primaria. Si `patient_id` tuviera
duplicados, `create_table` fallaría — la PK de una Feature Table debe ser única.

In [ ]:
n = features_df.count()
n_ids = features_df.select("patient_id").distinct().count()
print(f"Filas: {n} | patient_id únicos: {n_ids} | PK válida: {n == n_ids}")

nulos = features_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in features_df.columns
])
display(nulos)

## 4. Registro en Feature Store

Celda **idempotente**, requisito para que el Job se pueda reejecutar:

- Si la tabla no existe → `create_table` con `patient_id` como clave primaria.
- Si ya existe → `write_table(mode="merge")`, que actualiza filas existentes e inserta nuevas
  sin recrear la tabla (conserva lineage, permisos e historial).

In [ ]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

try:
    fe.get_table(name=FEATURE_TABLE)
    existe = True
except Exception:
    existe = False

if existe:
    fe.write_table(name=FEATURE_TABLE, df=features_df, mode="merge")
    print(f"🔄 Feature Table ACTUALIZADA por merge: {FEATURE_TABLE}")
else:
    fe.create_table(
        name=FEATURE_TABLE,
        primary_keys=["patient_id"],
        df=features_df,
        description=(
            "Features de tumores mamarios (Breast Cancer Wisconsin). "
            "6 variables 'mean' originales + 3 derivadas. "
            "Clave de entidad: patient_id. No contiene la etiqueta."
        ),
    )
    print(f"✅ Feature Table CREADA: {FEATURE_TABLE}")

## 5. Verificación (evidencia)

Leemos la Feature Table con el propio cliente — no con `spark.table()` — para demostrar
que está registrada en el Feature Store y no es sólo una tabla Delta más.

In [ ]:
fs_df = fe.read_table(name=FEATURE_TABLE)
print("Columnas en Feature Store:", fs_df.columns)
display(fs_df.select("patient_id", "radius", "area", "area_radius_ratio",
                     "perimeter_radius_ratio", "compactness_x_texture").limit(10))

print("Historial Delta de la Feature Table:")
display(spark.sql(f"DESCRIBE HISTORY {FEATURE_TABLE}").select("version", "timestamp", "operation"))

---
### Cierre notebook 02

```
cancer_raw (Delta)
   ├─ target ───────────────► se queda aquí (labels)
   └─ 6 features + 3 derivadas ──► Feature Table cancer_features (PK: patient_id)
```

**Evidencia para el examen:** en Catalog Explorer, `cancer_features` debe mostrar el
ícono/etiqueta de *Feature Table* y `patient_id` marcado como clave primaria.

Continúa con el notebook **03**.